# Section 01: Custom CNN models

### Data loading and preprocessing

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
from models.standard_cnn import StandardCNN
from models.lightweight_cnn import LightweightCNN
from data_prep import create_dataloaders
from utils import (
    count_parameters,
    train_model,
    evaluate_model,
    plot_training_history,
    plot_confusion_matrix,
    model_size_kb,
)

In [ ]:
DATASET_DIR = "./data/EuroSAT_RGB"

BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 0.001

SEED = 26
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
train_loader, val_loader, test_loader, class_names, mean, std = create_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    seed=SEED,
    image_size=64,
)

print("Training mean:", mean.tolist())
print("Training std: ", std.tolist())
print("Training:", len(train_loader.dataset))
print("Validation:", len(val_loader.dataset))
print("Testing:", len(test_loader.dataset))

### Model A: Standard CNN

In [ ]:
model_a = StandardCNN(num_classes=len(class_names)).to(DEVICE)
print(f"Model A trainable parameters: {count_parameters(model_a):,}")
print(f"Model A size: {model_size_kb(model_a):.2f} KB")

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model_a.parameters(),
    lr=LEARNING_RATE
)

history_a = train_model(
    model=model_a,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
)

In [ ]:
plot_training_history(history_a, model_name="Model A")

results_a = evaluate_model(model_a, test_loader, device=DEVICE)
print(f"Test Accuracy : {results_a['accuracy']:.4f}")
print(f"Test Precision: {results_a['precision']:.4f}")
print(f"Test Recall   : {results_a['recall']:.4f}")

plot_confusion_matrix(
    results_a["confusion_matrix"],
    class_names,
    model_name="Model A"
)

### Model B: Lightweight CNN

In [ ]:
model_b = LightweightCNN(num_classes=len(class_names)).to(DEVICE)
print(f"Model B trainable parameters: {count_parameters(model_b):,}")
print(f"Model B size: {model_size_kb(model_b):.2f} KB")

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model_b.parameters(),
    lr=LEARNING_RATE
)

history_b = train_model(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
)

In [ ]:
plot_training_history(history_a, model_name="Model A")
plot_training_history(history_b, model_name="Model B")

results_b = evaluate_model(model_b, test_loader, device=DEVICE)
print(f"Test Accuracy : {results_b['accuracy']:.4f}")
print(f"Test Precision: {results_b['precision']:.4f}")
print(f"Test Recall   : {results_b['recall']:.4f}")

plot_confusion_matrix(
    results_b["confusion_matrix"],
    class_names,
    model_name="Model B"
)